In [0]:
import pandas as pd

from pyspark.sql import functions as F
from datetime import datetime


In [0]:
# CELL 1 — Load config
UTILS_DIR = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/utils"
exec(open(f"{UTILS_DIR}/config.py").read())


print("✅ Config loaded")

In [0]:
# Show current schema of Bronze Products
# This is our baseline — the schema as it exists today
# We will add a new column and show the table absorbs it automatically

print("=== CURRENT SCHEMA — Bronze Products ===")
print("This is the schema before we simulate a new vendor feed arriving\n")

# printSchema() shows the structure of the table
# Each field shows: name, type, nullable
spark.table(BRONZE_PRODUCTS).printSchema()

print(f"\nCurrent column count: {len(spark.table(BRONZE_PRODUCTS).columns)}")
print(f"Current row count   : {spark.table(BRONZE_PRODUCTS).count()}")

In [0]:
# Simulate a new vendor feed with an extra column
# In real life, the vendor adds "product_color" to their Excel file
# Our pipeline receives this new feed and must handle it gracefully

# Read the existing products to simulate a realistic new feed
existing = spark.table(BRONZE_PRODUCTS).toPandas()

# Add a new column that didn't exist before
# This simulates the vendor adding "product_color" to their Excel file
existing["product_color"] = "Black"  # new column with a default value

print("=== SIMULATED NEW VENDOR FEED ===")
print(f"New column added: 'product_color'")
print(f"New column count: {len(existing.columns)}")
print("\nSample of new feed:")
print(existing[["product_id", "product_description_raw", "product_color"]].head(3))

In [0]:
# Write the new feed to the Bronze table
# mergeSchema=True tells Delta Lake to automatically add the new column
# WITHOUT this option, the write would FAIL with a schema mismatch error

print("=== WRITING NEW FEED WITH EXTRA COLUMN ===")

# Convert back to Spark DataFrame
new_feed_spark = spark.createDataFrame(existing)

# Write with mergeSchema=True — this is schema evolution in action
(new_feed_spark.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")   # ← enables schema evolution
    .saveAsTable(BRONZE_PRODUCTS))

print("✅ Write successful — new column absorbed automatically")

In [0]:
# Show the schema AFTER the new feed
# The table now has the new column
# Old data that didn't have this column will show null for product_color

print("=== SCHEMA AFTER NEW VENDOR FEED ===")
print("The new 'product_color' column was automatically added\n")

spark.table(BRONZE_PRODUCTS).printSchema()

print(f"\nNew column count: {len(spark.table(BRONZE_PRODUCTS).columns)}")
print(f"Row count preserved: {spark.table(BRONZE_PRODUCTS).count()}")

# Show the new column in the data
display(spark.table(BRONZE_PRODUCTS).select(
    "product_id",
    "product_description_raw",
    "product_color"   # ← new column now exists
).limit(5))

In [0]:
#  What would happen WITHOUT mergeSchema
# This cell demonstrates the error that schema evolution prevents
# We try to write with a new column but WITHOUT mergeSchema

print("=== WHAT HAPPENS WITHOUT mergeSchema ===")
print("Attempting to write new feed WITHOUT mergeSchema=True...\n")

try:
    # Add another new column to simulate the scenario
    existing["product_voltage"] = "110V"
    new_feed_spark2 = spark.createDataFrame(existing)

    # Try to write WITHOUT mergeSchema — this should FAIL
    (new_feed_spark2.write
        .format("delta")
        .mode("overwrite")
        # No mergeSchema option — will reject the new column
        .saveAsTable(BRONZE_PRODUCTS))

    print("❌ Write succeeded — schema evolution was not needed here")

except Exception as e:
    print("✅ Write FAILED as expected — without mergeSchema, new columns are rejected")
    print(f"\nError message: {str(e)[:200]}")
    print("\nThis is why mergeSchema=True is essential in production pipelines")

In [0]:
# CELL 7 — Restore the original schema
# Remove the demo columns to keep the pipeline clean
# In production, you would KEEP the new column — here we clean up for demo purposes

print("=== RESTORING ORIGINAL SCHEMA ===")
print("Removing demo columns added during schema evolution demonstration\n")

# Read current table and drop the demo columns
df_restored = spark.table(BRONZE_PRODUCTS).drop("product_color", "product_voltage")

# Write back with overwriteSchema to remove the extra columns
(df_restored.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  # ← removes columns, not just adds
    .saveAsTable(BRONZE_PRODUCTS))

print("✅ Original schema restored")
print(f"Column count restored to: {len(spark.table(BRONZE_PRODUCTS).columns)}")
spark.table(BRONZE_PRODUCTS).printSchema()